In [51]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [52]:
import sys, pathlib

# adjust this to your repo root if different
repo_root = pathlib.Path.cwd()

repo_root = repo_root.parent.parent

sys.path.insert(0, str(repo_root))
print('Inserted repo root into sys.path:', repo_root)

Inserted repo root into sys.path: /Users/fabian/qecsim


In [53]:
import pandas as pd
from scipy.interpolate import interp1d
import numpy as np
import glob

In [54]:
# Shared across poth types
DISTANCES = [3, 5, 7, 9]
N_SAMPLES  = 20_000
LABEL      = "CircuitNoise"
MARKERS    = ['o', 'v', '*', 's']
P_TEST     = np.array([2e-3, 2.5e-3, 3e-3])

In [55]:
# Loading in Gamma Values for memory and surgery
gamma_csv_memory = pd.read_csv(repo_root / "thesis" / "data" / "gamma_memory.csv")
gamma_csv_surgery = pd.read_csv(repo_root / "thesis" / "data" / "gamma_surgery.csv")

In [56]:
# Loading in all PTM matrices for memory and surgery
ptm_files_memory = glob.glob(str(repo_root / "thesis" / "data" / "ptm_memory_*.npy"))
ptm_files_surgery = glob.glob(str(repo_root / "thesis" / "data" / "ptm_surgery_*.npy"))

In [57]:
########## SURGERY ##########
# Perform basic interpolation and no fit like in the infdielity case
# -> Threshold crossover does not need to be determined
rows = []
for d, group in gamma_csv_surgery.groupby('distance'):
    sort_idx = np.argsort(group['physical_error_probability'].values)
    p_sorted = group['physical_error_probability'].values[sort_idx]
    g_sorted = group['gamma'].values[sort_idx]
    
    f = interp1d(p_sorted, g_sorted, kind='cubic')
    gamma_interp = f(P_TEST)
    
    for p, g in zip(P_TEST, gamma_interp):
        rows.append({'p': p, 'distance': d, 'gamma': round(g, 3), 'overhead': round(g**2, 2)})

# Create table for better visualization
table = (pd.DataFrame(rows)
           .pivot(index='p', columns='distance', values='gamma'))
table.index = [f"{p*100:.2f}%" for p in table.index]
table.columns = [f"d={d}" for d in table.columns]
print(table)

          d=3     d=5     d=7      d=9
0.20%   4.203   3.721   2.824    2.174
0.25%   8.489  10.431  10.044    8.161
0.30%  18.845  43.353  84.752  102.458


In [58]:
######### MEMORY #########
# Perform basic interpolation and no fit like in the infdielity case
# -> Threshold crossover does not need to be determined
rows = []
for d, group in gamma_csv_memory.groupby('distance'):
    sort_idx = np.argsort(group['physical_error_probability'].values)
    p_sorted = group['physical_error_probability'].values[sort_idx]
    g_sorted = group['gamma'].values[sort_idx]
    
    f = interp1d(p_sorted, g_sorted, kind='cubic')
    gamma_interp = f(P_TEST)
    
    for p, g in zip(P_TEST, gamma_interp):
        rows.append({'p': p, 'distance': d, 'gamma': round(g, 3), 'overhead': round(g**2, 2)})

# Create table for better visualization
table = (pd.DataFrame(rows)
           .pivot(index='p', columns='distance', values='gamma'))
table.index = [f"{p*100:.2f}%" for p in table.index]
table.columns = [f"d={d}" for d in table.columns]
print(table)

         d=3    d=5    d=7    d=9
0.20%  1.035  1.023  1.012  1.005
0.25%  1.057  1.043  1.029  1.017
0.30%  1.079  1.074  1.057  1.044


In [59]:
# Define functions for calculation of Qubits Counts
def qubits_patch(d):
    return (2*(d**2) - 1)

def qubits_surgery_addtional(d):
    return ((d - 1) / 2) + 2

In [60]:
# Getting all Qubit Budgets
for d in [3,5,7,9]:

    complete_qubits = (3 * qubits_patch(d)) + (qubits_surgery_addtional(d)) - ((d-1) / 2)

    print(f"Distance {d} needs {complete_qubits} qubits for the surgery approach")

Distance 3 needs 53.0 qubits for the surgery approach
Distance 5 needs 149.0 qubits for the surgery approach
Distance 7 needs 293.0 qubits for the surgery approach
Distance 9 needs 485.0 qubits for the surgery approach


In [61]:
qubits_patch(7)

97